In [ ]:
#!pip install flopy

# MODFLOW 6.

## Componentes.

**Véase [1]**.

<img src="../figures/components_mf6.png">

Una simulación típicamente se basa en cuatro componentes:
* **Models**. Un modelo resuelve un proceso hidrológico; por ejemplo, el GWF model, resuelve la ecuación de flujo subterráneo usando el método CVFD.
* **Exchange**. Facilita la comunicación entre dos modelos; por ejemplo, un intercambio GWF-GWF, permite que algunas celdas de un modelo GWF estén hidráulicamente conectadas con celdas del otro modelo GWF.
* **Solutions**. Resuelve uno o más modelos hidrológicos, utiliza métodos iterativos para resolver sistemas no-lineales.
* **Timing**. Controla el paso de tiempo y determina el fin de una simulación.

**<font color='Green'>[1] Langevin, C.D., Hughes, J.D., Provost, A.M., Banta, E.R., Niswonger, R.G., and Panday, Sorab, 2017, Documentation for the MODFLOW 6 Groundwater Flow (GWF) Model: U.S. Geological Survey Techniques and Methods, book 6, chap. A55, 197 p., accessed August 4, 2017. </font>** https://doi.org/10.3133/tm6A55.

## Esquema para GWF.

<img src="../figures/modflow01.png" width=300px hspace="5" vspace="5" style="float: left;"/>

**Tomado de [2]**.

Figura 2. Diagrama esquemático que muestra varias formas comunes en las que los modelos de flujo de agua subterránea (GWF) y transporte de agua subterránea (GWT) se pueden configurar dentro de una simulación. El caso de uso común representado en (A) consiste en un único modelo GWF que se resuelve mediante una solución numérica.

* El modelo GWF descrito en [3] se divide en *paquetes*, como se hacía en versiones anteriores de MODFLOW.
* Un paquete es la parte del modelo que se ocupa de un único aspecto de la simulación.
* Por ejemplo, el paquete *Well* simula el efecto de los pozos y el paquete *River* simula el efecto de los ríos.
* El Modelo GWF contiene muchos paquetes y opciones que el usuario puede o no tener ocasión de utilizar.

**<font color='Green'>[2] Langevin, C. D., Hughes, J. D., Provost, A. M., Russcher, M. J., & Panday, S. (2023). MODFLOW as a configurable Multi‐Model Hydrologic Simulator. Ground Water.</font>** https://doi.org/10.1111/gwat.13351

# Proceso de solución de flujo en 1D.

**Con base en**: 

**<font color='Green'>[3] MODFLOW 6 – Example problems, MODFLOW 6 Development Team, with contributions from Chieh Ying Chen and Mike Toews 02/07/2024.**</font>  **32 MOC3D Problem 1**. (Archivo: `mf6examples.pdf` del directorio `doc` de la distribución de MODFLOW 6).


## Paso 0. Importación de bibliotecas

In [2]:
import os, sys     # Intefaces con el sistema operativo.
import numpy as np # Manejo de arreglos numéricos multidimensionales
import matplotlib.pyplot as plt # Graficación

# Biblioteca y módulos de flopy
import flopy
from flopy.plot.styles import styles

# Extras para mf6 y flopy (módulo creado en este proyecto)
import xmf6

In [3]:
init = {
    'sim_name' : "flow",
    'exe_name' : "C:\\Users\\luiggi\\Documents\\GitSites\\xmf6\\mf6\\windows\\mf6",
#    'exe_name' : "../../mf6/macosarm/mf6",
    'sim_ws' : "sandbox1"
}

time = {
    'units': "seconds",
    'nper' : 1,
    'perioddata': [(120.0, 1, 1.0)]
}

ims = {}

gwf = { 
    'modelname': init["sim_name"],
    'model_nam_file': f"{init["sim_name"]}.nam",
    'save_flows': True
}

o_sim = xmf6.gwf.initialize(silent = False, init = init, time = time, ims = ims)



sim configuration
――――――――――――――――――
sim_name = flow
exe_name = C:\Users\luiggi\Documents\GitSites\xmf6\mf6\windows\mf6
  sim_ws = sandbox1
――――――――――――――――――


time configuration
―――――――――――――――――――
     units = seconds
      nper = 1
perioddata = ―― data array ――
           = (120.0, 1, 1.0)
―――――――――――――――――――


numerical solution configuration
―――――――――――――――――――――――――――――――――
―――――――――――――――――――――――――――――――――


In [7]:
dis = {
    'length_units' : "centimeters",
    'nlay': 1, 
    'nrow': 1, 
    'ncol': 120,
    'delr': 0.1, 
    'delc': 0.1,
    'top' : 1.0, 
    'botm': 0.0 
}

ic = {
    'strt': 1.0
}

npf = {
    'save_specific_discharge': True,
    'save_saturation' : True,
    'icelltype' : 0,
    'k' : 0.01,
}

chd = {
    'stress_period_data': [[(0, 0, dis['ncol'] - 1), 1.0]],     
}

## Physical parameters
specific_discharge = 0.1  # Specific discharge ($cm s^{-1}$)
source_concentration = 1.0  # Source concentration (unitless)
q   = specific_discharge * dis['delc'] * dis['delr'] * dis['top']
aux = source_concentration
print(aux)
well = {
    'stress_period_data': [[(0, 0, 0), q, aux,]],
    'pname': "WEL-1",
#    'save_flows': True,
    'auxiliary' : ["CONCENTRATION"],

}

oc = {
    'budget_filerecord': f"{init['sim_name']}.bud",
    'head_filerecord': f"{init['sim_name']}.hds",
    'saverecord' : [("HEAD", "ALL"), ("BUDGET", "ALL")],

}

o_gwf = xmf6.gwf.build(o_sim, silent = True,
                       gwf = gwf, dis = dis, ic = ic, chd = chd, npf = npf, oc = oc, well = well)

o_sim.write_simulation(silent = True)

1.0


In [5]:
o_sim.run_simulation()

FloPy is using the following executable to run the model: ..\..\..\mf6\windows\mf6.exe
                                   MODFLOW 6
                U.S. GEOLOGICAL SURVEY MODULAR HYDROLOGIC MODEL
                            VERSION 6.6.1 02/10/2025

   MODFLOW 6 compiled Feb 10 2025 17:37:25 with Intel(R) Fortran Intel(R) 64
   Compiler Classic for applications running on Intel(R) 64, Version 2021.7.0
                             Build 20220726_000000

This software has been approved for release by the U.S. Geological 
Survey (USGS). Although the software has been subjected to rigorous 
review, the USGS reserves the right to update the software as needed 
pursuant to further analysis and review. No warranty, expressed or 
implied, is made by the USGS or the U.S. Government as to the 
functionality of the software and related material nor shall the 
fact of release constitute any such warranty. Furthermore, the 
software is released on condition that neither the USGS nor the U.S. 
Gover

(True, [])

In [6]:
# Obtenemos los resultados de la carga hidráulica
head = flopy.utils.HeadFile(
    os.path.join(init['sim_ws'], 
                 oc['head_filerecord'])).get_data()

# Obtenemos los resultados del BUDGET
bud  = flopy.utils.CellBudgetFile(
    os.path.join(init['sim_ws'], 
                 oc['budget_filerecord']),
    precision='double'
)
# Obtenemos las velocidades
spdis = bud.get_data(text='DATA-SPDIS')[0]
qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, o_gwf)

# Verificamos el tipo y dimensiones de los arreglos donde
# están almacenados la carga hidráulica, el BUDGET, y la velocidad.
print('Head : ', type(head), head.shape)
print('Budget : ', type(bud), bud.shape)
print('spdis : ', type(spdis), spdis.shape)
print('qx : ', type(qx), qx.shape)
print('qy : ', type(qy), qy.shape)
print('qz : ', type(qz), qz.shape)

print(head.shape, '\n', head)
print(qx)

Head :  <class 'numpy.ndarray'> (1, 1, 120)
Budget :  <class 'flopy.utils.binaryfile.CellBudgetFile'> (np.int32(1), np.int32(1), np.int32(120))
spdis :  <class 'numpy.rec.recarray'> (120,)
qx :  <class 'numpy.ndarray'> (1, 1, 120)
qy :  <class 'numpy.ndarray'> (1, 1, 120)
qz :  <class 'numpy.ndarray'> (1, 1, 120)
(1, 1, 120) 
 [[[12.9 12.8 12.7 12.6 12.5 12.4 12.3 12.2 12.1 12.  11.9 11.8 11.7 11.6
   11.5 11.4 11.3 11.2 11.1 11.  10.9 10.8 10.7 10.6 10.5 10.4 10.3 10.2
   10.1 10.   9.9  9.8  9.7  9.6  9.5  9.4  9.3  9.2  9.1  9.   8.9  8.8
    8.7  8.6  8.5  8.4  8.3  8.2  8.1  8.   7.9  7.8  7.7  7.6  7.5  7.4
    7.3  7.2  7.1  7.   6.9  6.8  6.7  6.6  6.5  6.4  6.3  6.2  6.1  6.
    5.9  5.8  5.7  5.6  5.5  5.4  5.3  5.2  5.1  5.   4.9  4.8  4.7  4.6
    4.5  4.4  4.3  4.2  4.1  4.   3.9  3.8  3.7  3.6  3.5  3.4  3.3  3.2
    3.1  3.   2.9  2.8  2.7  2.6  2.5  2.4  2.3  2.2  2.1  2.   1.9  1.8
    1.7  1.6  1.5  1.4  1.3  1.2  1.1  1. ]]]
[[[0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01

In [ ]:
# Obtenemos el objeto con la información de la discretización espacial
grid = o_gwf.modelgrid

# Coordenadas del centro de celdas para graficación
x, _, _ = grid.xyzcellcenters

with styles.USGSPlot():
    plt.rcParams['font.family'] = 'DeJavu Sans'
    plt.figure(figsize=(10,3))
    plt.plot(x[0], head[0, 0], marker=".", ls ="-", mec="blue", mfc="none", markersize="1", label = 'Head')
    plt.xlim(0, 12)
    plt.xticks(ticks=np.linspace(0, grid.extent[1],13))
    plt.xlabel("Distance (cm)")
    plt.ylabel("Head")
    plt.legend()
    plt.grid()
    plt.show()

## Paso 1. Definición de parámetros del problema.

### Parámetros para la discretización del espacio.

<img src="../figures/mesh_plainview_mf6.png" width=300px hspace="5" vspace="5" style="float: right;"/>
<img src="../figures/mesh_3D_mf6.png" width=300px hspace="5" vspace="5" style="float: right;"/>

* El modelo de malla consiste de 1 capa, 120 columnas y 1 renglón.
* La longitud del renglón es de 12 [cm].
* La longitud de la columna es 0.1 [cm].
* Con la información anterior se calcula el ancho del renglón, DELC, y de las columnas, DELR, que ambos casos debe ser 0.1 $cm$.
* La parte superior (TOP) de la celda es 1.0 [cm] y la parte inferior (BOTTOM) es cero.
* La longitud de la capa es igual a 1.0 [cm], valor que se calcula de |TOP - BOTTOM|.

<img src="../figures/flow_mf6.png">

|Parameter | Value| Units | Variable |
|---:|:---:|:---:|:---|
|Length of system (rows) |12.0| cm | `mesh.row_length` |
|Number of layers |1| | `mesh.nlay` |
|Number of rows |1| | `mesh.nrow` |
|Number of columns |120| | `mesh.ncol` |
|Column width |0.1| cm | `mesh.delr` |
|Row width |0.1| cm | `mesh.delc`|
|Top of the model |1.0| cm | `mesh.top`|
|Layer bottom elevation (cm) |0| cm | `mesh.bottom` |

In [ ]:
dis = {
    'length_units' : "centimeters",
    'nlay': 1, 
    'nrow': 1, 
    'ncol': 120,
    'delr': 1.0, 
    'delc': 1.0,
    'top' : 1.0, 
    'botm': 0.0 
}

xmf6.nice_print(dis, 'Space discretization')

### Parámetros para la discretización del tiempo.

<center>
<img src="../figures/time_step.png" width=500px>
</center>
  
The length of a time step is calculated by multiplying the length of the previous time step by TSMULT. 
The length of the first time step, $\Delta t_1$, is related to PERLEN, NSTP, and TSMULT by the relation:
$$
\Delta t_1= \frac{\text{PERLEN}}{\text{NSTP}} \;\; \text{para} \;\; \text{TSMULT} = 1
$$

$$
\Delta t_1= \text{PERLEN} \frac{\text{TSMULT} - 1}{\text{TSMULT}^\text{nstp}-1} \;\; \text{para} \;\; \text{TSMULT} \neq 1
$$

The length of each successive time step is computed as

$$
\Delta t = \Delta t_{old} \text{TSMULT}
$$

where:
* `perlen` (double) is the length of a stress period.
* `nstp` (integer) is the number of time steps in a stress period.
* `tsmult` (double) is the multiplier for the length of successive time steps.
  
**<font color='Green'>[4] Hughes, J.D., Langevin, C.D., and Banta, E.R., 2017, *Documentation for the MODFLOW 6 framework: U.S. Geological Survey Techniques and Methods*, book 6, chap. A57, 40 p.,</font>** https://doi.org/10.3133/tm6A57. **Timing Module, pp 10,12**.

|Parameter | Value| Units | Variable |
|---:|:---:|:---:|:---|
|Number of stress periods |1| | `tm_par['nper']` |
|Total time |120| s | `tm_par['total_time']` |
|Number of time steps| 1 | | `tm_par['nstp']`|
|Multiplier | 1 | | `tm_par['tsmult']`|

In [ ]:
time = {
    'units': "seconds",
    'nper' : 1,
    'perioddata': [(120.0, 1, 1.0)]
}

xmf6.nice_print(time, 'Time discretization')

### Parámetros físicos.

|Parameter | Value| Units | Variable |
|---:|:---:|:---:|:---|
|Specific discharge |0.1| cm s$^{-1}$ | `ph_par['specific_discharge']` |
|Hydraulic conductivity |0.01| cm s$^{-1}$ | `ph_par['hydraulic_conductivity']` |
|Source concentration |1.0| unitless | `ph_par['source_concentration']` |


In [ ]:
## Physical parameters
specific_discharge = 0.1  # Specific discharge ($cm s^{-1}$)
source_concentration = 1.0  # Source concentration (unitless)
q   = specific_discharge * dis['delc'] * dis['delr'] * dis['top']
aux = source_concentration

## Paso 2. MODFLOW6 environment y salida.

In [ ]:
init = {
    'sim_name' : "flow",
    'exe_name' : "C:\\Users\\luiggi\\Documents\\GitSites\\xmf6\\mf6\\windows\\mf6",
#    'exe_name' : "../../mf6/macosarm/mf6",
    'sim_ws' : "sandbox1"
}

oc = {
    'budget_filerecord': f"{init['sim_name']}.bud",
    'head_filerecord': f"{init['sim_name']}.hds",
    'saverecord' : [("HEAD", "ALL"), ("BUDGET", "ALL")],

}

xmf6.nice_print(init, 'MODFLOW 6 environment')

xmf6.nice_print(oc, 'Output files')

## Paso 3. Definición de la simulación (`MFSimulation` object)

**Recordemos que:**

<img src="../figures/modflow01.png" width=300px hspace="5" vspace="5" style="float: left;"/>

In [ ]:
o_sim = flopy.mf6.MFSimulation(
    sim_name = init['sim_name'], 
    sim_ws   = init['sim_ws'], 
    exe_name = init['exe_name']
)
print(o_sim)

## Paso 4. Definición de la discretización temporal (`ModflowTDis` object)

In [ ]:
flopy.mf6.ModflowTdis(
    o_sim, 
    nper = time['nper'], 
    perioddata = time['perioddata'], 
    time_units = time["units"]
)

## Paso 5. Definición de la solución numérica (`ModflowIms` object)

In [ ]:
ims = flopy.mf6.ModflowIms(o_sim)
print(ims)

## Paso 6. Modelo de Flujo (ModflowGwf object)

### GWF: Paquetes

**<font color='Green'>Véase [1].</font>**

<img src="../figures/gwf_mf6.png" width=500px hspace="5" vspace="5" style="float: left;"/>
<img src="../figures/gwf_mf6_pack.png" width=500px hspace="5" vspace="5" style="float: left;"/>

In [ ]:
o_gwf = flopy.mf6.ModflowGwf(
    o_sim, 
    modelname  = init['sim_name'], 
    save_flows = True
)
print(o_gwf)

## Paso 7. Paquete: discretización espacial (`ModflowGwfdis` object)

In [ ]:
mesh = flopy.mf6.ModflowGwfdis(
    o_gwf,
    length_units = dis["length_units"],
    nlay = dis['nlay'],
    nrow = dis['nrow'],
    ncol = dis['ncol'],
    delr = dis['delr'],
    delc = dis['delc'],
    top  = dis['top'],
    botm = dis['botm'],
)
print(mesh)

## Paso 8. Paquete: condiciones iniciales (`ModflowGwfic` object)



In [ ]:
ic = flopy.mf6.ModflowGwfic(
    o_gwf, 
    strt=1.0 # Initial head
) 
print(ic)

## Paso 9. Paquete: propiedades de flujo en los nodos (`ModflowGwfnpd` object)

In [ ]:
npf = flopy.mf6.ModflowGwfnpf(
    o_gwf,
    save_specific_discharge = True,
    save_saturation = True,
    icelltype = 0,
    k = 0.01,
)
print(npf)

## Paso 10. Paquete: CHD (`ModflowGwfchd` object)

In [ ]:
chd = flopy.mf6.ModflowGwfchd(
    o_gwf, 
    stress_period_data=[[(0, 0, dis['ncol'] - 1), 1.0]]  # Node, Constant value
) 
print(chd)

## Paso 11. Paquete: Pozos (`ModflowGwfwel` object)

In [ ]:
wel = flopy.mf6.ModflowGwfwel(
    o_gwf,
    stress_period_data = [[(0, 0, 0), q, aux,]],
    pname = "WEL-1",
    auxiliary = ["CONCENTRATION"],
)

print(wel)

## Paso 12. Paquete: salida (`ModflowGwfoc` object)

In [ ]:
oc = {
    'budget_filerecord': f"{init['sim_name']}.bud",
    'head_filerecord': f"{init['sim_name']}.hds",
    'saverecord' : [("HEAD", "ALL"), ("BUDGET", "ALL")],

}

oc = flopy.mf6.ModflowGwfoc(
    o_gwf,
    head_filerecord   = oc['head_filerecord'],
    budget_filerecord = oc['budget_filerecord'],
    saverecord = [("HEAD", "ALL"), ("BUDGET", "ALL")],
)
print(oc)

## Paso 13. Escritura de los archivos de entrada para MODFLOW 6

In [ ]:
sim.write_simulation()

## Paso 14. Ejecución de la simulación.

In [ ]:
sim.run_simulation()

## Paso 15. Postprocessing

In [ ]:
# Obtenemos los resultados de la carga hidráulica
head = flopy.utils.HeadFile(
    os.path.join(os_par['ws'], 
                 oc_par['head_file'])).get_data()

# Obtenemos los resultados del BUDGET
bud  = flopy.utils.CellBudgetFile(
    os.path.join(os_par['ws'], 
                 oc_par['fbudget_file']),
    precision='double'
)
# Obtenemos las velocidades
spdis = bud.get_data(text='DATA-SPDIS')[0]
qx, qy, qz = flopy.utils.postprocessing.get_specific_discharge(spdis, gwf)

In [ ]:
# Verificamos el tipo y dimensiones de los arreglos donde
# están almacenados la carga hidráulica, el BUDGET, y la velocidad.
print('Head : ', type(head), head.shape)
print('Budget : ', type(bud), bud.shape)
print('spdis : ', type(spdis), spdis.shape)
print('qx : ', type(qx), qx.shape)
print('qy : ', type(qy), qy.shape)
print('qz : ', type(qz), qz.shape)

In [ ]:
print(head.shape, '\n', head)

In [ ]:
print(qx.shape, '\n', qx)

In [ ]:
with styles.USGSPlot():
    plt.rcParams['font.family'] = 'DeJavu Sans'
    x, _, _ = mesh.get_coords()
    plt.figure(figsize=(10,3))
    plt.plot(x, head[0, 0], marker=".", ls ="-", mec="blue", mfc="none", markersize="1", label = 'Head')
    plt.xlim(0, 12)
    plt.xticks(ticks=np.linspace(0, mesh.row_length,13))
    plt.xlabel("Distance (cm)")
    plt.ylabel("Head")
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
plt.figure(figsize=(10,0.15))
ax = plt.gca()
pmv0 = flopy.plot.PlotMapView(gwf, ax=ax)
pmv0.plot_grid(colors='dimgray', lw=0.5)
plt.yticks(ticks=[0, 0.1],fontsize=8)

plt.show()


# 